In [1]:
import numpy as np
import os
import glob
import xarray as xr
from scipy import interpolate
from netCDF4 import Dataset
from datetime import datetime
import pandas as pd
import time
import matplotlib.pyplot as plt

data_dir = '/home/lukeew/Local_work/data/01_SLW_xgb'
vdr_mask_data_dir = data_dir + '/CHCH_v6_reference/preprocessed{}'
output_dir = data_dir + '/CHCH_v6_reference/processed{}/'


liq_threshold = 0.1
mix_threshold = 0.4


In [2]:
date = '20210518'

In [3]:
def classify_bin(ds, liq_threshold = liq_threshold, mix_threshold = mix_threshold):

    ds['undet_nan']   = xr.where((ds.vdr.isnull()),1,0).astype('int8') # Gets Nans due to missing data

    # Applies (0,1) threshold for VDR
    ds['vdr']  = xr.where(ds.vdr<0,np.nan,ds.vdr)
    ds['vdr']  = xr.where(ds.vdr>1,np.nan,ds.vdr)

    ds['bin_wlwc'] = xr.where((ds.cloud_mask == 1) & (ds.vdr <= liq_threshold) & (ds.temp >= 0),1,0).astype('int8') # Warm liquid water clouds
    ds['bin_slwc'] = xr.where((ds.cloud_mask == 1) & (ds.vdr <= liq_threshold) & (ds.temp <  0),1,0).astype('int8') # Supercooled liquid water clouds
    ds['bin_mpc'] = xr.where((ds.cloud_mask == 1) & (ds.vdr <= mix_threshold) & (ds.vdr > liq_threshold) & (ds.temp <  0),1,0).astype('int8') # Mixed phase clouds
    ds['bin_slcc'] = xr.where((ds.cloud_mask == 1) & (ds.vdr <= mix_threshold) & (ds.temp <  0),1,0).astype('int8') # Supercooled liquid water containing clouds 
    ds['bin_ic']   = xr.where((ds.cloud_mask == 1) & (ds.vdr >  mix_threshold) & (ds.temp <  0),1,0).astype('int8') # Ice clouds

    ds['bin_liq'] = xr.where((ds.bin_wlwc) | (ds.bin_slwc),1,0).astype('int8')

    ds['bin_undet']   = xr.where((ds.cloud_mask == 1) & (ds.vdr > liq_threshold) & (ds.temp > 0),1,0).astype('int8') # Undetermined from VDR and temperatures alone

    return ds

In [4]:
infile = vdr_mask_data_dir.format(date[:6]) + '/{}.nc'.format(date)
print('  <--- ' + infile)

ds = xr.open_dataset(infile)

ds = classify_bin(ds)


  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v6_reference/preprocessed202105/20210518.nc


In [56]:
ds

<xarray.Dataset>
Dimensions:      (time: 288, level: 300)
Coordinates:
  * time         (time) datetime64[ns] 2021-05-18T00:02:29.999991 ... 2021-05...
  * level        (level) float64 25.0 75.0 125.0 ... 1.492e+04 1.498e+04
Data variables: (12/14)
    copol        (time, level) float64 ...
    crosspol     (time, level) float64 ...
    cloud_mask   (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    backscatter  (time, level) float64 ...
    temp         (time, level) float64 9.723 9.723 9.07 ... -57.87 -57.86 -57.85
    vdr          (time, level) float64 1.304 0.02969 0.02929 ... 0.5166 -28.96
    ...           ...
    bin_slwc     (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    bin_mpc      (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    bin_slcc     (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    bin_ic       (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    bin_liq      (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    bin_undet    (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0

In [32]:
x = [0,0,0,0,0,0,0,0,0,1,0,1,0]
y = [0,0,0,0,0,0,0,0,0,1,1,1,0]
a = [0,0,1,1,1,1,1,0,1,1,1,1,0]

cloud_mask = a

# layer_int(x, a)

cm_df = pd.Series(cloud_mask)
x_df = pd.Series(x)
y_df = pd.Series(y)

heights_x_arr = np.full(np.shape(cloud_mask),np.nan)  # array of length (n altitude bins)

# print(len(cm_df[cm_df==1].groupby((cm_df==0).cumsum())))

# print((cm_df==0).cumsum())
idx = 0
for i,v in cm_df[cm_df==1].groupby((cm_df==0).cumsum()):
    n_x = x_df[v.index].sum()
    n_y = y_df[v.index].sum()

    counts = np.array([n_x,n_y])

    print(np.argmax(counts))

    # layers_x_arr[idx] = x_layer_int
    # heights_x_arr[v.index] = x_layer_int
    # idx += 1

# layers_x_arr,heights_x_arr

0
1


In [5]:
def layer(cloud_mask, liq, mix, ice, undet):

    cm_df = pd.Series(cloud_mask)
    liq_df = pd.Series(liq)
    mix_df = pd.Series(mix)
    ice_df = pd.Series(ice)
    undet_df = pd.Series(undet)

    heights_x_arr = np.full(np.shape(cloud_mask),np.nan)  # array of length (n altitude bins)

    for _,v in cm_df[cm_df==1].groupby((cm_df==0).cumsum()):
        n_liq = liq_df[v.index].sum()
        n_mix = mix_df[v.index].sum()
        n_ice = ice_df[v.index].sum()
        n_undet = undet_df[v.index].sum()

        counts = np.array([n_undet,n_mix,n_liq,n_ice])  #order important because if two n are the same, the first will be chosen (so undetermined the best)
        mode = np.argmax(counts)
        heights_x_arr[v.index] = mode

    return heights_x_arr

In [17]:
ds.isel(time=0)

<xarray.Dataset>
Dimensions:      (level: 300)
Coordinates:
    time         datetime64[ns] 2021-05-18T00:02:29.999991
  * level        (level) float64 25.0 75.0 125.0 ... 1.492e+04 1.498e+04
Data variables: (12/14)
    copol        (level) float64 ...
    crosspol     (level) float64 ...
    cloud_mask   (level) int8 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0
    backscatter  (level) float64 ...
    temp         (level) float64 9.723 9.723 9.07 8.258 ... -51.55 -51.58 -51.62
    vdr          (level) float64 nan 0.02969 0.02929 0.02735 ... 0.3621 nan nan
    ...           ...
    bin_slwc     (level) int8 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0
    bin_mpc      (level) int8 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0
    bin_slcc     (level) int8 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0
    bin_ic       (level) int8 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0
    bin_liq      (level) int8 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0
    bin_undet    (level) int8 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0

In [6]:
modes_all = np.full(np.shape(ds.cloud_mask),np.nan)

for time_idx, cm_profile in enumerate(ds.cloud_mask):
    ds_time =  ds.isel(time=time_idx)  
    modes = layer(ds_time.cloud_mask.values, ds_time.bin_liq.values, ds_time.bin_mpc.values, ds_time.bin_ic.values, ds_time.bin_undet.values)
    # modes = layer(ds.cloud_mask[time_idx].values, ds.bin_liq[time_idx].values, ds.bin_mpc[time_idx].values, ds.bin_ic[time_idx].values, ds.bin_undet[time_idx].values)
    modes_all[time_idx] = modes

modes_da = xr.DataArray(modes_all, coords=[('time',ds.time.data),('level',ds.level.data)])

ds['layer_ice'] = xr.where((modes_da == 3),1,0).astype('int8')
ds['layer_liq'] = xr.where((modes_da == 2),1,0).astype('int8')
ds['layer_mix'] = xr.where((modes_da == 1),1,0).astype('int8')
ds['layer_undet'] = xr.where((modes_da == 0),1,0).astype('int8')

ds['layer_wlwc'] = xr.where((modes_da == 2) & (ds.temp >= 0),1,0).astype('int8')
ds['layer_slwc'] = xr.where((modes_da == 2) & (ds.temp < 0),1,0).astype('int8')

In [7]:
ds

<xarray.Dataset>
Dimensions:      (time: 288, level: 300)
Coordinates:
  * time         (time) datetime64[ns] 2021-05-18T00:02:29.999991 ... 2021-05...
  * level        (level) float64 25.0 75.0 125.0 ... 1.492e+04 1.498e+04
Data variables: (12/20)
    copol        (time, level) float64 ...
    crosspol     (time, level) float64 ...
    cloud_mask   (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    backscatter  (time, level) float64 ...
    temp         (time, level) float64 9.723 9.723 9.07 ... -57.87 -57.86 -57.85
    vdr          (time, level) float64 nan 0.02969 0.02929 ... nan 0.5166 nan
    ...           ...
    layer_ice    (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    layer_liq    (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    layer_mix    (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    layer_undet  (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    layer_wlwc   (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    layer_slwc   (time, level) int8 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0

In [29]:
ds['layer_ice'].max()

<xarray.DataArray 'layer_ice' ()>
array(1, dtype=int8)

In [40]:
print(ds.copol[28])
ds['copol'] =  xr.where((ds.copol == -999),np.nan,ds.copol).astype('int8')
print(ds.copol[28])

<xarray.DataArray 'copol' (height: 300)>
array([-999., -999., -999., ..., -999., -999., -999.])
Coordinates:
    time     datetime64[ns] 2021-05-18T02:22:29.999973
  * height   (height) float64 25.0 75.0 125.0 ... 1.488e+04 1.492e+04 1.498e+04
<xarray.DataArray 'copol' (height: 300)>
array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [47]:
ds.undet_nan.sum()

<xarray.DataArray 'undet_nan' ()>
array(900)

In [48]:
ds['vdr']  = xr.where(ds.vdr<0,np.nan,ds.vdr)
ds['vdr']  = xr.where(ds.vdr>1,np.nan,ds.vdr)
ds['undet_nan']   = xr.where((ds.vdr.isnull()),1,0).astype('int8') # Data missing


In [49]:
ds.undet_nan.sum()

<xarray.DataArray 'undet_nan' ()>
array(41372)

In [38]:
ds

new_da = xr.DataArray(np.ones(np.shape(ds.cloud_mask)), coords=[ds.time, ds.height], dims=['time', 'level'])
    
    # ('time',ds.time.data), ('level',ds.height.data)])
new_da
# ds['new'] = new_da
# ds

<xarray.DataArray (time: 288, level: 300)>
array([[1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       ...,
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.]])
Coordinates:
  * time     (time) datetime64[ns] 2021-05-18T00:02:29.999991 ... 2021-05-18T...
  * level    (level) float64 25.0 75.0 125.0 ... 1.488e+04 1.492e+04 1.498e+04

In [8]:
outfile = output_dir.format(date[:-2])+str(date)+'.nc'
ds.to_netcdf(outfile, format='NETCDF4')